In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('data\cleaned_dataset.csv')
df.head()

,Gesture,Ax,Ay,Az,Gx,Gy,Gz
0,0,0.327250,0.972020,0.978133,0.972500,0.282577,0.551446
1,0,0.323264,0.978824,0.978574,0.968821,0.297214,0.549468
2,0,0.331779,0.970411,0.978970,0.970301,0.260290,0.551673
3,0,0.313128,0.981825,0.978850,0.968926,0.284897,0.547513
4,0,0.222645,0.968425,0.979011,0.969466,0.267591,0.550981


In [3]:
X = df[['Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz']]
y = df['Gesture']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42)

In [5]:
y_train.head()

93     0
69     0
203    1
171    1
152    1
Name: Gesture, dtype: int64

In [6]:
X_train.head()

,Ax,Ay,Az,Gx,Gy,Gz
93,0.373526,0.959517,0.978514,0.968943,0.273717,0.549192
69,0.294803,0.949733,0.945690,0.967886,0.267014,0.549612
203,0.335536,0.854595,0.978258,0.967506,0.278506,0.553692
171,0.291664,0.899851,0.977883,0.967934,0.279439,0.550018
152,0.255149,0.891572,0.978456,0.967804,0.249687,0.555387


In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True)
lr_schedual = LearningRateScheduler(lambda epoch: 1e-3 * 0.9 ** epoch)

In [13]:
history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_data=(
    X_test, y_test), callbacks=(early_stopping, lr_schedual))

Epoch 1/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.5028 - loss: 0.7981 - val_accuracy: 0.6364 - val_loss: 0.7124 - learning_rate: 0.0010
Epoch 2/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6354 - loss: 0.6626 - val_accuracy: 0.7879 - val_loss: 0.6937 - learning_rate: 9.0000e-04
Epoch 3/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6354 - loss: 0.7280 - val_accuracy: 0.7576 - val_loss: 0.6752 - learning_rate: 8.1000e-04
Epoch 4/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6796 - loss: 0.6417 - val_accuracy: 0.7879 - val_loss: 0.6654 - learning_rate: 7.2900e-04
Epoch 5/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6409 - loss: 0.6128 - val_accuracy: 0.8182 - val_loss: 0.6558 - learning_rate: 6.5610e-04
Epoch 6/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6851 - loss: 0.6584 - val_accuracy: 0.8182 - val_loss: 0.6498 - learning_rate: 5.9049e-04
Epoch 7/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6851 - loss: 0.5

In [14]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'test_loss: {test_loss}, test_accuracy: {test_accuracy}')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8182 - loss: 0.4543
test_loss: 0.454288512468338, test_accuracy: 0.8181818127632141


In [15]:
y_hat = model.predict(X_test)
y_hat = [0 if val < 0.5 else 1 for val in y_hat]

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step


In [19]:
print(y_train.value_counts())

Gesture
0    91
1    90
Name: count, dtype: int64


In [20]:
accuracy_score(y_test, y_hat)

0.8181818181818182

In [21]:
model.save('tfModel.keras')

In [22]:
del model

In [23]:
model = load_model('tfModel.keras')

In [24]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'test_loss: {test_loss:.4f}')
print(f'test_accuracy: {test_accuracy * 100:.2f}%')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8182 - loss: 0.4543 
test_loss: 0.4543
test_accuracy: 81.82%
